# 매일 알아서 업데이트되는 한국 전력수급 대시보드 만들기

지난 세션에서는 **"이미 만들어진 Parquet + DuckDB-Wasm 대시보드가 어떻게 돌아가는가"**를 봤습니다.
오늘은 그 앞단, **"그 재료(Parquet)를 누가 매일 사다 놓는가"**를 직접 만들어 봅니다.

| 순서 | 내용 | 시간 |
|---|---|---|
| Part 1 | 사이트가 데이터를 받아오는 주소 찾기 (개발자도구) | 10분 |
| Part 2 | 실습 ① 일별 전력수급실적 받아오기 | 20분 |
| Part 3 | 실습 ② 월별 연료원별 전력거래량 받아오기 (+ 공식 API 체험) | 15분 |
| Part 4 | 실습 ③ Parquet 로 저장하고 DuckDB 로 SQL 실행 | 15분 |
| Part 5 | 실습 ④ 브라우저 대시보드에 연결하기 | 15분 |
| Part 6 | 실습 ⑤ GitHub Actions 로 자동화하기 | 10분 |
| Part 7 | 정리: 어떤 데이터에 자동화가 맞을까 | 5분 |

**오늘 쓰는 데이터** (둘 다 한국전력거래소 EPSIS, `epsis.kpx.or.kr`)

* **전력수급실적(일별)**: 매일 전일 실적이 올라옵니다. 최대전력, 공급능력, 공급예비율 등. → *메인*
* **전력거래량 연료원별(월별)**: 매월 말 한 달 전 실적이 올라옵니다. 원자력·석탄·LNG·신재생 등. → *보조*

In [ ]:
!pip -q install duckdb

import re, time, json, os
from datetime import date, datetime, timedelta

import duckdb
import pandas as pd
import requests

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

EPSIS = "https://epsis.kpx.or.kr/epsisnew"
HEADERS = {"User-Agent": "energy-dashboard-workshop/1.0 (educational)"}

# EPSIS가 Colab(해외 서버)에서 막혀 있을 때만 쓰는 비상용 주소.
# 강사가 배포한 대시보드 주소 뒤에 /data/ 를 붙여 넣으세요. 예) "https://내사이트.netlify.app/data/"
BACKUP_BASE = ""

## Part 1. 사이트가 데이터를 받아오는 주소 찾기

EPSIS 의 **전력수급실적** 화면(`전력수급 → 전력수급실적`)에는 조회 화면과 `EXCEL`·`CSV` 버튼만 있고,
**공공데이터포털에 API 가 없습니다.** 하지만 화면에 표가 뜬다는 것은, 화면이 어딘가에서 데이터를 받아온다는 뜻입니다.

**같이 해보기** (Chrome 기준)

1. 주소 `https://epsis.kpx.or.kr/epsisnew/selectEkgeEpsMepChart.do?menuId=030100` 를 엽니다.
2. `F12` (맥은 `⌘ + ⌥ + I`) → **Network** 탭 → 필터에서 **Fetch/XHR** 선택
3. 화면에서 기간을 바꿔 **조회**를 누릅니다.
4. 새로 생긴 요청 `selectEkgeEpsMep.ajax` 를 클릭합니다.
   * **Headers**: 요청 방식은 `POST`
   * **Payload**: `beginDate=20261020`, `endDate=20261028`, `selYear`, `selMonth`
   * **Response**: JSON 이 아니라 `gridData.push({...})` 라는 자바스크립트 코드 😮

> 💡 **공식 API vs 화면 뒤 주소**
>
> | | 공식 API (공공데이터포털) | 화면 뒤 주소 (오늘의 일별 데이터) |
> |---|---|---|
> | 인증키 | 필요 | 불필요 |
> | 문서 | 있음 | 없음 (직접 찾아야 함) |
> | 응답 | JSON/XML | 사이트 마음대로 (오늘은 자바스크립트 코드) |
> | 안정성 | 높음 | **사이트가 개편되면 깨질 수 있음** |
>
> 공식 API 가 있으면 그걸 쓰는 것이 원칙입니다. 없을 때의 차선책이라는 점을 기억하세요.

## Part 2. 실습 ① 일별 전력수급실적 받아오기

### 2-1. 요청을 직접 보내보기
브라우저가 한 일을 파이썬 `requests` 로 똑같이 합니다. 일단 **3일치만** 받아서 응답이 어떻게 생겼는지 봅니다.

In [ ]:
# EPSIS가 막힐 때를 대비한 예비 응답(2026-10-26~28, 3일치). 정상이면 쓰이지 않습니다.
SAMPLE_DAILY_RAW = r"""
var c1,c2,c3,c4,c5,c6,c8,c9;
var count = $("#selCount option:selected").val();
var point = "";
if(count == 0) point = "";
else if(count == 1) point = ".0";
else if(count == 2) point = ".00";
else if(count == 3) point = ".000";
gridData = [];


c1 = textFormmat("161945.0",count);
c3 = textFormmat("92522.0",count);
c4 = textFormmat("70319.0",count);
c5 = textFormmat("22203.0",count);
c6 = textFormmat("31.6",count);
c7 = textFormmat("2026/10/28(19:00)",count);
c8 = ("48411.0" == 0) ? "-" : Math.round("48411.0").toString().replace(/\B(?=(\d{3})+(?!\d))/g, ",") + point;
c9 = ("2026/10/28(13:00)" == "") ? "-" : textFormmat("2026/10/28(13:00)",count);

gridData.push({"year":"2026",
"month":"10",
"day":"28",
"c1":c1,
"c3":c3,
"c4":c4,
"c8":c8,
"c9":c9,
"c5":c5,
"c6":c6,
"c7":c7
});

c1 = textFormmat("161918.0",count);
c3 = textFormmat("94309.0",count);
c4 = textFormmat("70637.0",count);
c5 = textFormmat("23672.0",count);
c6 = textFormmat("33.5",count);
c7 = textFormmat("2026/10/27(19:00)",count);
c8 = ("45713.0" == 0) ? "-" : Math.round("45713.0").toString().replace(/\B(?=(\d{3})+(?!\d))/g, ",") + point;
c9 = ("2026/10/27(13:00)" == "") ? "-" : textFormmat("2026/10/27(13:00)",count);

gridData.push({"year":"2026",
"month":"10",
"day":"27",
"c1":c1,
"c3":c3,
"c4":c4,
"c8":c8,
"c9":c9,
"c5":c5,
"c6":c6,
"c7":c7
});

c1 = textFormmat("161918.0",count);
c3 = textFormmat("88685.0",count);
c4 = textFormmat("70932.0",count);
c5 = textFormmat("17753.0",count);
c6 = textFormmat("25",count);
c7 = textFormmat("2026/10/26(19:00)",count);
c8 = ("48702.0" == 0) ? "-" : Math.round("48702.0").toString().replace(/\B(?=(\d{3})+(?!\d))/g, ",") + point;
c9 = ("2026/10/26(04:00)" == "") ? "-" : textFormmat("2026/10/26(04:00)",count);

gridData.push({"year":"2026",
"month":"10",
"day":"26",
"c1":c1,
"c3":c3,
"c4":c4,
"c8":c8,
"c9":c9,
"c5":c5,
"c6":c6,
"c7":c7
});
"""

def epsis_post(path, data, retries=3):
    """EPSIS에 POST 요청을 보내고 본문 텍스트를 돌려준다. 실패하면 잠깐 쉬었다 다시 시도."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(f"{EPSIS}/{path}", data=data, headers=HEADERS, timeout=60)
            r.raise_for_status()
            return r.text
        except requests.RequestException as e:
            last = e
            time.sleep(2 ** i)
    raise RuntimeError(f"EPSIS 요청 실패: {last}")

try:
    raw = epsis_post("selectEkgeEpsMep.ajax",
                     {"beginDate": "20261026", "endDate": "20261028", "selYear": "", "selMonth": ""})
    EPSIS_OK = True
except RuntimeError as e:
    print("⚠️", e)
    print("→ 예비 응답으로 파싱 연습만 진행합니다. 이후 전체 기간은 BACKUP_BASE 로 불러옵니다.")
    raw = SAMPLE_DAILY_RAW
    EPSIS_OK = False

print(len(raw), "글자")
print("\n".join(l for l in raw.splitlines() if l.strip())[:1200])

### 2-2. 자바스크립트 코드에서 값 뽑아내기
응답은 아래 두 가지 패턴의 반복입니다.

```
c4 = textFormmat("70319.0",count);          ← ① 값이 변수에 들어가고
gridData.push({"year":"2026", "month":"10", "day":"28", "c4":c4, ...});   ← ② 변수 이름으로 한 줄이 만들어진다
```

그래서 **정규식(regex) 두 개**면 충분합니다.
(정규식이 낯설면 Claude 에게 "이 응답에서 값을 뽑는 정규식을 만들어줘"라고 시키고, **결과를 검증**하는 것이 이 실습의 핵심입니다.)

In [ ]:
_ASSIGN = re.compile(r'\b(c\d+|localnm|year)\s*=\s*(?:textFormmat\()?\(?"([^"]*)"')   # c4 = textFormmat("70319.0", ...)
_PAIR   = re.compile(r'"(\w+)"\s*:\s*(?:"([^"]*)"|(\w+))')                           # "year":"2026"  또는  "c4":c4

def parse_grid(text):
    """응답 속 gridData.push(...) 블록을 dict 리스트로 바꾼다."""
    parts = re.split(r"gridData\.push\(", text)
    rows, pending = [], parts[0]           # 첫 블록의 값 대입문은 첫 push 앞에 있다
    for part in parts[1:]:
        body, _, after = part.partition("})")
        values = dict(_ASSIGN.findall(pending))
        row = {}
        for m in _PAIR.finditer(body):
            key, literal, var = m.group(1), m.group(2), m.group(3)
            row[key] = literal if literal is not None else values.get(var)
        rows.append(row)
        pending = after                    # 다음 블록의 값 대입문은 이번 push 뒤에 있다
    if not rows:
        raise RuntimeError("데이터를 찾지 못했습니다. 사이트 구조가 바뀌었는지 확인하세요.")
    return rows

pd.DataFrame(parse_grid(raw))

열 이름이 `c1, c3, c4…` 로 되어 있습니다. 화면의 표 머리글과 맞춰 보면 (개발자도구 > Elements 에서 확인 가능):

| 코드 | 의미 | 단위 | | 코드 | 의미 | 단위 |
|---|---|---|---|---|---|---|
| c1 | 설비용량 | MW | | c5 | 공급예비력 | MW |
| c3 | 공급능력 | MW | | c6 | 공급예비율 | % |
| c4 | 최대전력 | MW | | c7 | 최대전력 기준일시 | |
| c8 | 최소전력 | MW | | c9 | 최소전력 기준일시 | |

### 2-3. 정리해서 DataFrame 으로 만들고, 전체 기간 받기
열 이름을 알기 쉽게 바꾸고, 숫자·날짜 형식으로 바꿔서 **함수**로 묶습니다. (이 함수가 곧 자동화에 쓸 `fetch_daily.py` 입니다.)

In [ ]:
COLUMNS = {"c1": "capacity_mw", "c3": "supply_mw", "c4": "peak_mw", "c8": "min_mw", "c5": "reserve_mw", "c6": "reserve_pct"}

def _hour(text):
    """'2026/10/28(19:00)' -> 19"""
    m = re.search(r"\((\d{1,2}):", text or "")
    return int(m.group(1)) if m else None

def to_daily(rows):
    df = pd.DataFrame(rows)
    out = pd.DataFrame({"date": pd.to_datetime(df["year"] + df["month"] + df["day"], format="%Y%m%d").dt.date})
    for src, dst in COLUMNS.items():
        out[dst] = pd.to_numeric(df[src], errors="coerce")
    out["peak_hour"] = df["c7"].map(_hour).astype("Int64")
    out["min_hour"] = df["c9"].map(_hour).astype("Int64")
    out.loc[out["min_mw"] == 0, "min_mw"] = None     # 0 은 '값 없음'
    return out.sort_values("date").reset_index(drop=True)

def fetch_daily(begin, end):
    text = epsis_post("selectEkgeEpsMep.ajax", {"beginDate": begin.strftime("%Y%m%d"), "endDate": end.strftime("%Y%m%d"),
                                                "selYear": "", "selMonth": ""})
    return to_daily(parse_grid(text))

to_daily(parse_grid(raw))

In [ ]:
START, END = date(2022, 1, 1), date.today()

if EPSIS_OK:
    daily_raw = fetch_daily(START, END)          # 요청은 딱 1번
else:
    assert BACKUP_BASE, "BACKUP_BASE 에 강사가 알려준 주소를 넣고 이 셀을 다시 실행하세요."
    daily_raw = pd.read_parquet(BACKUP_BASE + "daily_supply.parquet")

print(f"{len(daily_raw):,}행, {daily_raw['date'].min()} ~ {daily_raw['date'].max()}")
daily_raw.tail()

### 2-4. ⚠️ 받은 데이터를 그냥 믿지 않기 (검증)
**자동화가 가장 위험한 순간은 "틀린 데이터가 조용히 쌓일 때"입니다.** 사람이 눈으로 보던 검토를 코드로 옮겨 놓아야 합니다.
아래 세 가지를 확인해 봅니다: ① 날짜가 빠짐없는가 ② 같은 날짜가 두 번 들어 있지 않은가 ③ 값이 상식 범위인가

In [ ]:
expected_days = (daily_raw["date"].max() - daily_raw["date"].min()).days + 1
dups = daily_raw["date"].duplicated().sum()

print("받은 행 수      :", len(daily_raw))
print("있어야 할 일수  :", expected_days)
print("중복된 날짜     :", dups)
print("최대전력 범위   :", daily_raw["peak_mw"].min(), "~", daily_raw["peak_mw"].max(), "MW")

if dups:
    print("\n🔍 어느 날짜가 중복인가?")
    display(daily_raw[daily_raw["date"].duplicated(keep=False)].head(6))

> 실제로 원본에는 **2023-04-01 ~ 04-10 열흘치가 똑같은 값으로 두 번** 들어 있습니다. 모르고 `GROUP BY` 하면 그 달의 평균·합계가 틀어집니다.
> 값까지 완전히 같은 행이라 한 줄을 지워도 안전합니다. (값이 *다른* 중복이라면 어느 쪽이 맞는지 사람이 판단해야 합니다.)

In [ ]:
daily = daily_raw.drop_duplicates().reset_index(drop=True)
assert not daily["date"].duplicated().any(), "값이 다른 중복이 남아 있습니다 - 사람이 확인해야 합니다"
assert len(daily) == (daily["date"].max() - daily["date"].min()).days + 1, "빠진 날짜가 있습니다"
assert daily["peak_mw"].between(30_000, 120_000).all(), "상식 범위를 벗어난 최대전력이 있습니다"
print("✅ 검증 통과:", len(daily), "일")

### ✏️ 연습 2-A  최대전력이 가장 높았던 날 5개를 찾아보세요 (pandas)
힌트: `sort_values`, `head`

In [ ]:
# 여기에 써보세요

<details><summary>정답 보기</summary>

`daily.nlargest(5, "peak_mw")[["date", "peak_mw", "peak_hour", "reserve_pct"]]`

</details>

In [ ]:
# 정답 (막히면 실행해 보세요)
daily.nlargest(5, "peak_mw")[["date", "peak_mw", "peak_hour", "reserve_pct"]]

### 2-5. 🤖 Claude 에게 시키면? — 두 가지 방법 비교
같은 일을 Claude 에게 두 가지 방식으로 시켜서 결과를 비교해 봅니다. (2~3명씩 짝을 지어 서로 다른 방식을 시도)

| | 방법 A: 직접 긁어오게 하기 | 방법 B: 코드를 짜게 하기 |
|---|---|---|
| 프롬프트 예 | "EPSIS 전력수급실적에서 2026년 10월 최대전력을 일별로 알려줘" | "EPSIS 전력수급실적(일별)을 받는 파이썬 코드를 짜줘. 데이터 주소는 `selectEkgeEpsMep.ajax`, 요청 파라미터는 beginDate, endDate …" |
| 결과 | 숫자가 텍스트로 나옴 | 코드가 나옴 → **내가 실행** |
| 검증 | 숫자를 하나하나 눈으로 대조해야 함 | 위 2-4 의 검증 코드를 그대로 붙일 수 있음 |
| 다음 달에도? | 다시 시켜야 함 | **그대로 다시 실행**하면 됨 |

**확인해 볼 것:** Claude 가 알려준 10월 최대전력 중 하나를 골라, 위 `daily` 표의 값과 같은지 대조해 보세요.

## Part 3. 실습 ② 월별 연료원별 전력거래량 받아오기

같은 방법을 다른 메뉴에 적용합니다. `전력거래 → 전력거래량 → 연료원별` 화면의 데이터 주소는 `selectEkmaPtdBft.ajax` 이고,
요청 값은 `selYear=N`(월별), `selRegion=1`(전국 합계) 입니다. 이 메뉴는 **한 번 요청하면 2002년 1월부터 전체 기간**이 옵니다.

In [ ]:
text = epsis_post("selectEkmaPtdBft.ajax", {"selYear": "N", "selRegion": "1"}) if EPSIS_OK else None

if text:
    wide = pd.DataFrame(parse_grid(text))
else:
    assert BACKUP_BASE, "BACKUP_BASE 가 필요합니다."
    wide = None

if wide is not None:
    print(wide.shape)
    display(wide[["Period", "c1", "c23", "c4", "c5", "c6", "c912", "c7", "c9", "c10"]].head(3))

열 의미 (화면 표 머리글 기준, 단위 **GWh**): `c1` 원자력 · `c23` 석탄(유연탄+무연탄) · `c4` 유류 · `c5` LNG · `c6` 양수 · `c912` 신재생 계 · `c7` 기타 · `c9` PPA(태양광) · `c10` 전체 합계

**넓은 표(wide) → 긴 표(long)**: 연료원이 열로 늘어서 있는 표를 `월 · 연료원 · 값` 3열로 바꾸면 SQL 로 다루기 훨씬 쉽습니다.

In [ ]:
FUELS = {"c1": "원자력", "c23": "석탄", "c4": "유류", "c5": "LNG", "c6": "양수", "c912": "신재생", "c7": "기타", "c9": "PPA(태양광)"}

def to_monthly(wide):
    wide = wide.copy()
    wide["month"] = pd.to_datetime(wide["Period"] + "/01", format="%Y/%m/%d").dt.date
    long = (wide.melt(id_vars="month", value_vars=list(FUELS), var_name="col", value_name="gwh")
                .assign(fuel=lambda d: d["col"].map(FUELS), gwh=lambda d: pd.to_numeric(d["gwh"], errors="coerce"))
                .drop(columns="col").dropna(subset=["gwh"])
                .sort_values(["month", "fuel"]).reset_index(drop=True))
    return long[["month", "fuel", "gwh"]]

if wide is not None:
    monthly = to_monthly(wide)
else:
    monthly = pd.read_parquet(BACKUP_BASE + "monthly_fuel.parquet")

# 검증: 연료원 합이 '전체 합계(c10)'와 같은가? (표를 잘못 읽었는지 가장 빨리 알 수 있는 방법)
if wide is not None:
    latest = wide.iloc[0]
    print("가장 최근 월:", latest["Period"], "| 연료원 합:", round(monthly[monthly["month"] == monthly["month"].max()]["gwh"].sum(), 1),
          "| 화면의 전체 합계(c10):", round(float(latest["c10"]), 1))
print(f"{len(monthly):,}행, {monthly['month'].min()} ~ {monthly['month'].max()}")
monthly.tail(8)

In [ ]:
# 한눈에 보기: 최근 5년 연료원별 비중 (연도별)
m = monthly.assign(year=pd.to_datetime(monthly["month"]).dt.year)
share = m[m["year"] >= 2021].pivot_table(index="year", columns="fuel", values="gwh", aggfunc="sum")
(share.div(share.sum(axis=1), axis=0) * 100).round(1)

### 🏛️ 덤: 공식 Open API 도 한 번 불러보기 (인증키가 있는 사람만)
같은 연료원별 거래량은 **공공데이터포털 「한국전력거래소_전력거래실적」** 에서 공식 API 로도 제공됩니다.
사전 준비: [data.go.kr/data/15099766](https://www.data.go.kr/data/15099766/openapi.do) → **활용신청**(개발계정은 자동 승인, 하루 100회) → 마이페이지에서 **일반 인증키(Decoding)** 복사

* 공식 API 는 `tradeDay`(거래일자)를 넣어 **날짜·시간별 거래량**을 줍니다. 정산 기준이라 **약 2개월 전 실적까지**만 제공됩니다.
* 응답의 항목: `fuel`(연료원), `tradeDay`, `time`, `pcap`(설비용량 MW), `mgo`(전력거래량 MWh), `rtotal`(전력거래대금)
* 키 없는 사람은 이 셀을 건너뛰어도 됩니다. 키는 **노트북에 직접 쓰지 말고** 입력창이나 환경변수로 받으세요.

In [ ]:
import getpass

try:
    api_key = os.environ.get("KPX_API_KEY") or getpass.getpass("공공데이터포털 인증키(Decoding) - 없으면 그냥 Enter: ")
except (EOFError, KeyboardInterrupt):
    api_key = ""

if api_key:
    r = requests.get(
        "https://apis.data.go.kr/B552115/PowerTradingResultInfo1/getPowerTradingResultInfo1",
        params={"serviceKey": api_key, "numOfRows": 10, "pageNo": 1, "dataType": "json", "tradeDay": "20260801"},
        timeout=60,
    )
    print(r.status_code)
    print(r.text[:1500])      # 응답이 없으면 tradeDay 를 바꿔보세요 (정산 기준 약 2개월 전까지만 제공)
else:
    print("인증키가 없어 건너뜁니다.")

## Part 4. 실습 ③ Parquet 로 저장하고 DuckDB 로 SQL 실행

지난 세션의 내용을 직접 해봅니다. 같은 데이터를 CSV 와 Parquet 로 저장해서 비교하고, 발표자료 5쪽의 **pandas ↔ SQL 대응**을 실제로 실행합니다.

In [ ]:
import os, time

os.makedirs("out", exist_ok=True)
daily.to_csv("out/daily_supply.csv", index=False)
daily.to_parquet("out/daily_supply.parquet", index=False)
monthly.to_parquet("out/monthly_fuel.parquet", index=False)

for f in ["daily_supply.csv", "daily_supply.parquet", "monthly_fuel.parquet"]:
    print(f"{f:24s} {os.path.getsize('out/' + f) / 1024:8.1f} KB")

> 💬 **솔직한 메모:** 일별 데이터는 1,700행 남짓이라 CSV 와 Parquet 의 차이가 크지 않습니다.
> 지난 세션의 실태조사 원자료(6.6만 행 × 247열)에서 용량이 1/6, 읽기 속도가 약 20배 빨라지던 효과는 **행·열이 많을 때** 드러납니다.
> 이번 실습에서 Parquet 를 쓰는 더 중요한 이유는 **날짜·숫자 같은 자료형이 파일에 기억된다**는 점입니다.

In [ ]:
con = duckdb.connect()
con.sql("CREATE VIEW daily   AS SELECT * FROM 'out/daily_supply.parquet'")
con.sql("CREATE VIEW monthly AS SELECT * FROM 'out/monthly_fuel.parquet'")

con.sql("DESCRIBE daily").df()[["column_name", "column_type"]]

### pandas 와 SQL 은 같은 일을 다른 말로 하는 것
질문: **"연도별로 평균 최대전력을 구해줘"**

In [ ]:
# pandas
pd_result = (daily.assign(연도=pd.to_datetime(daily["date"]).dt.year)
                  .groupby("연도")["peak_mw"].mean().round())

# SQL (DuckDB)
sql_result = con.sql("""
    SELECT year(date) AS 연도, round(avg(peak_mw)) AS peak_mw
    FROM daily
    GROUP BY 1
    ORDER BY 1
""").df()

display(pd_result.to_frame())
display(sql_result)

### ✏️ 연습 4-A  여름철(7~8월) 평균 최대전력은 해마다 얼마인가요?
힌트: `WHERE month(date) IN (7, 8)`

In [ ]:
con.sql("""
    -- 여기에 써보세요
""").df()

<details><summary>정답 보기</summary>

```sql
SELECT year(date) AS 연도, round(avg(peak_mw)) AS 여름_평균_최대전력
FROM daily
WHERE month(date) IN (7, 8)
GROUP BY 1
ORDER BY 1
```

</details>

In [ ]:
# 정답 (막히면 실행해 보세요)
con.sql("""
    SELECT year(date) AS 연도, round(avg(peak_mw)) AS 여름_평균_최대전력
    FROM daily
    WHERE month(date) IN (7, 8)
    GROUP BY 1
    ORDER BY 1
""").df()

### ✏️ 연습 4-B  두 데이터를 월 단위로 연결하기 (JOIN)
일별 표와 월별 표는 주기가 다릅니다. 일별을 **월평균**으로 묶으면 월별 표와 `JOIN` 할 수 있습니다.
**월평균 최대전력(daily)** 과 **그 달 신재생 비중(monthly)** 을 나란히 보는 표를 만들어 보세요. 힌트: `date_trunc('month', date)`

In [ ]:
con.sql("""
    -- 여기에 써보세요
""").df()

<details><summary>정답 보기</summary>

```sql
WITH d AS (SELECT date_trunc('month', date) AS m, avg(peak_mw) AS avg_peak FROM daily GROUP BY 1),
     f AS (SELECT month AS m,
                  100 * sum(gwh) FILTER (WHERE fuel = '신재생') / sum(gwh) AS re_pct
           FROM monthly GROUP BY 1)
SELECT strftime(d.m, '%Y-%m') AS 월, round(d.avg_peak) AS 월평균_최대전력, round(f.re_pct, 1) AS 신재생_비중
FROM d JOIN f USING (m)
ORDER BY 1 DESC
LIMIT 12
```

</details>

In [ ]:
# 정답 (막히면 실행해 보세요)
con.sql("""
    WITH d AS (SELECT date_trunc('month', date) AS m, avg(peak_mw) AS avg_peak FROM daily GROUP BY 1),
         f AS (SELECT month AS m,
                      100 * sum(gwh) FILTER (WHERE fuel = '신재생') / sum(gwh) AS re_pct
               FROM monthly GROUP BY 1)
    SELECT strftime(d.m, '%Y-%m') AS 월, round(d.avg_peak) AS 월평균_최대전력, round(f.re_pct, 1) AS 신재생_비중
    FROM d JOIN f USING (m)
    ORDER BY 1 DESC
    LIMIT 12
""").df()

## Part 5. 실습 ④ 브라우저 대시보드에 연결하기

지금까지 만든 `out/daily_supply.parquet`, `out/monthly_fuel.parquet` 두 파일이 곧 **대시보드의 재료**입니다.
강사가 준비한 대시보드 템플릿(`site/index.html`)은 지난 세션의 흐름 그대로 동작합니다.

```
브라우저가 링크 접속 → DuckDB-Wasm 내려받기 → Parquet 2개 내려받기 → SQL 실행 → Plotly 그래프
```

템플릿의 핵심은 이 몇 줄뿐입니다 (`site/index.html` 의 `boot()`):

```js
const bundle = await duckdb.selectBundle(duckdb.getJsDelivrBundles());     // ① 엔진
...
const buf = new Uint8Array(await (await fetch("data/daily_supply.parquet")).arrayBuffer());   // ② 데이터
await db.registerFileBuffer("daily_supply.parquet", buf);
await conn.query("CREATE VIEW daily AS SELECT * FROM read_parquet('daily_supply.parquet')");  // ③ 표 등록
const result = await conn.query("SELECT ... FROM daily ...");                                  // ④ SQL
```

**해보기**
1. 아래 셀로 내 Parquet 를 내려받습니다.
2. 템플릿 폴더의 `site/data/` 에 덮어씁니다.
3. 로컬에서 열기: 터미널에서 `python3 -m http.server --directory site 8000` → 브라우저에서 `http://localhost:8000`
4. 대시보드 맨 아래 **"직접 SQL 써보기"** 에서 방금 연습한 SQL 을 실행해 봅니다. (표 이름은 `daily`, `monthly`)

In [ ]:
try:
    from google.colab import files
    for f in ["daily_supply.parquet", "monthly_fuel.parquet"]:
        files.download(f"out/{f}")
except ImportError:
    print("Colab 이 아니므로 out/ 폴더의 파일을 그대로 쓰면 됩니다.")

## Part 6. 실습 ⑤ GitHub Actions 로 자동화하기

지금까지는 **내가 이 노트북을 실행해야** 데이터가 갱신됩니다. 이 일을 GitHub 서버가 대신하게 만듭니다.

```
[매일 08:00] GitHub Actions 가 깨어남
   → fetch_daily.py 실행 (오늘 만든 코드와 같은 로직)
   → site/data/daily_supply.parquet 이 바뀌었으면 커밋·푸시
   → Netlify 가 푸시를 감지해서 사이트를 다시 배포
   → 링크를 여는 사람은 항상 최신 데이터를 봄
```

워크플로 파일은 `.github/workflows/daily.yml` 입니다. 핵심만 읽어 봅니다.

```yaml
on:
  schedule:
    - cron: "0 23 * * *"     # ← 언제? 분 시 일 월 요일. UTC 기준이라 23:00 UTC = 다음 날 08:00 KST
  workflow_dispatch:         # ← 버튼으로 수동 실행도 가능

permissions:
  contents: write            # ← 결과를 저장소에 커밋할 권한

steps:
  - uses: actions/checkout@v4                 # 저장소 내려받기
  - uses: actions/setup-python@v5             # 파이썬 설치
  - run: pip install -r requirements.txt
  - run: python scripts/fetch_daily.py        # ← 오늘 만든 코드
  - run: |                                    # 바뀐 게 있을 때만 커밋
      git add site/data
      git diff --cached --quiet || (git commit -m "data: 갱신" && git push)
```

**해보기**
1. 템플릿 저장소를 내 GitHub 로 가져오기 (Fork 또는 Use this template)
2. 저장소의 **Actions** 탭 → `daily-supply` → **Run workflow** 버튼
3. 초록색 체크가 뜨면 → 저장소의 `site/data/` 에 새 커밋이 생겼는지 확인
4. (Netlify 연결된 경우) 잠시 뒤 사이트의 "데이터 수집" 시각이 바뀌는지 확인

**자주 막히는 곳**
* ❌ **빨간 X**: Actions 탭에서 해당 실행을 눌러 로그를 읽으세요. 대부분 마지막 줄에 이유가 있습니다.
* 🔒 **push 권한 오류**: 저장소 *Settings → Actions → General → Workflow permissions* 가 `Read and write` 인지 확인
* 🌏 **EPSIS 접속 실패**: GitHub 서버는 해외에 있습니다. 해외 IP 를 막는 사이트라면 실패할 수 있으니 **먼저 수동 실행으로 확인**하세요.
* 💤 공개 저장소는 **60일간 활동이 없으면 예약 실행이 꺼질 수** 있습니다. 가끔 Actions 탭을 확인하세요.
* 🔑 공식 API 를 쓰게 되면 인증키는 코드에 쓰지 말고 *Settings → Secrets and variables → Actions* 에 저장한 뒤 `env: KPX_API_KEY: ${{ secrets.KPX_API_KEY }}` 로 전달합니다.

## Part 7. 정리 — 어떤 데이터에 자동화가 맞을까

| | 자동화가 잘 맞는 데이터 | 사람 검토가 먼저인 데이터 |
|---|---|---|
| 예 | 일별 전력수급, 환율, 유가 | 연 1회 나오는 실태조사 원자료 |
| 형식 | 열 구성이 매번 똑같음 | 해마다 문항·코드값이 바뀜 |
| 실수의 비용 | 오늘 값이 틀려도 내일 바로잡힘 | 한 번 틀리면 1년 동안 그대로 |
| 방법 | 수집 → **검증 코드** → 저장을 자동화 | 변환은 코드로, **코드값 비교는 사람이** |

**오늘의 체크리스트**
* ✅ 공식 API 가 있는지 먼저 찾는다 (공공데이터포털) → 없으면 화면 뒤 주소를 찾는다
* ✅ Claude 에게 "긁어와"보다 **"코드를 짜줘"** 를 시킨다 → 같은 코드를 계속 쓸 수 있고, 검증할 수 있다
* ✅ 받자마자 **검증**한다 (날짜 빠짐·중복·상식 범위·합계 일치)
* ✅ 호출은 **하루 1회, 필요한 기간만**. 출처(한국전력거래소 EPSIS)를 화면에 적는다
* ✅ 사이트가 바뀌면 깨진다 → 빈 응답이면 **조용히 넘어가지 말고 오류로 멈추게** 한다 (그래야 Actions 가 빨간 X 로 알려준다)

**더 해볼 거리**
* 같은 방법으로 `발전·판매 → 발전량 → 에너지원별`, `계통한계가격 → 시간별 SMP`(매일 갱신) 붙이기
* 대시보드에 지역(`selRegion`)별 연료원 비교 추가하기
* 기후(기온) 데이터를 붙여서 "기온이 올라가면 최대전력이 얼마나 오르는가" 보기